# H2-1 확장 — 7단계: 다중비교 보정 (Benjamini-Hochberg)

**6단계 결과 요약** (raw p-value 기준)
- 유의(p<0.05): TypeC×고지출(coef=-6.25, p=0.0004), TypeB×고지출(coef=-2.43, p=0.0118)
- 나머지 7개 조합: 유의하지 않음 (TypeC×저지출은 계수는 크지만 n=28, p=0.163으로 노이즈 가능성)

**이번 단계**: Type×계층 9개 조합을 한꺼번에 검정했으므로, 개별 p-value를 그대로 "유의하다"고
판단하면 다중비교 문제(false positive 확률 증가)가 생김. Benjamini-Hochberg(FDR) 보정을 적용해
"9개 중 몇 개가 보정 후에도 살아남는지" 확정.

**예상(6단계 해석 시 암산)**: TypeC×고지출만 살아남고, TypeB×고지출은 근소하게 탈락할 가능성.
이번 단계에서 정확히 계산해 확정.


## 0. 환경 설정

In [1]:
from importlib.metadata import version
print("linearmodels", version("linearmodels"), "| statsmodels", version("statsmodels"))

linearmodels 7.0 | statsmodels 0.14.6


In [2]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS
from statsmodels.stats.multitest import multipletests

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

from config import (DATA_DIR, OUT_DIR, INTERMEDIATE_DIR, FIG_DIR, SEED, TIER_MIN_WEEK, TIER_MAX_WEEK,
                    CAMP_MIN_WEEK, CAMP_MAX_WEEK, MID_WEEK, CELL_SIZE_THRESH, TIER_EN,
                    load_tiers, load_panel, load_camp_full)



## 1~6단계 파이프라인 재구성 (동일 로직 — 새 내용 없음)

In [3]:
panel = load_panel()
tier_df = load_tiers()

# --- Type x 계층 상호작용 더미 ---
for t in ["TypeA", "TypeB", "TypeC"]:
    for tier in ["저지출", "중지출", "고지출"]:
        panel[f"{t}_x_{tier}"] = ((panel[f"active_{t}"] == 1) & (panel["tier3"] == tier)).astype(int)

interaction_cols = [f"{t}_x_{tier}" for t in ["TypeA", "TypeB", "TypeC"] for tier in ["저지출", "중지출", "고지출"]]

# --- 셀 크기표 (4단계) ---
def cell_size_table(tier_col):
    out = {}
    for t in ["TypeA", "TypeB", "TypeC"]:
        hh_active = panel.loc[panel[f"active_{t}"] == 1, ["household_key", tier_col]].drop_duplicates()
        out[t] = hh_active.groupby(tier_col, observed=True)["household_key"].nunique()
    return pd.DataFrame(out)
cell3 = cell_size_table("tier3")

# --- 6단계 본 회귀 ---
reg_df = panel.copy()
reg_df["entity"] = reg_df["household_key"]
reg_df["time"] = reg_df["WEEK_NO"]
reg_df = reg_df.set_index(["entity", "time"])

model = PanelOLS(
    dependent=reg_df["spend"],
    exog=reg_df[interaction_cols],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
result = model.fit(cov_type="clustered", cluster_entity=True)

print("패널 로드 + 6단계 회귀 재실행 완료")
print(result.params)

패널 로드 + 6단계 회귀 재실행 완료
TypeA_x_저지출    1.056302
TypeA_x_중지출    0.819589
TypeA_x_고지출   -0.162506
TypeB_x_저지출    1.442018
TypeB_x_중지출    0.417724
TypeB_x_고지출   -2.429732
TypeC_x_저지출    9.675478
TypeC_x_중지출   -1.219311
TypeC_x_고지출   -6.252591
Name: parameter, dtype: float64


## 7단계 — 다중비교 보정 (Benjamini-Hochberg / FDR)

9개 조합의 p-value를 한꺼번에 검정했으므로, 개별 p<0.05만으로 유의성을 판단하지 않고
FDR(허위발견율) 5% 기준으로 보정.

In [4]:
coef_table = result.params.to_frame("coef")
coef_table["se"] = result.std_errors
coef_table["t"] = result.tstats
coef_table["p_raw"] = result.pvalues
coef_table = coef_table.reset_index().rename(columns={"index": "변수"})
coef_table[["Type", "계층"]] = coef_table["변수"].str.split("_x_", expand=True)

# 셀 크기 병합
cell_long = cell3.stack().reset_index()
cell_long.columns = ["계층", "Type", "수신가구수"]
coef_table = coef_table.merge(cell_long, on=["Type", "계층"], how="left")
coef_table["표본주의"] = coef_table["수신가구수"] < CELL_SIZE_THRESH

# --- BH(FDR) 보정 ---
reject, p_adj, _, _ = multipletests(coef_table["p_raw"], alpha=0.05, method="fdr_bh")
coef_table["p_adj_bh"] = p_adj
coef_table["유의_BH보정후"] = reject

_pt = pd.read_csv(OUT_DIR / "pretrend_tests.csv").set_index("scope")
pretrend_note = {t: f"5단계 p={_pt.loc[TIER_EN[t], 'p_value']:.3f} (미수신 {int(_pt.loc[TIER_EN[t], 'n_never_receive']):,}가구)" for t in ["저지출", "중지출", "고지출"]}
coef_table["평행추세_참고"] = coef_table["계층"].map(pretrend_note)

cols_order = ["Type", "계층", "coef", "se", "p_raw", "p_adj_bh", "유의_BH보정후", "수신가구수", "표본주의", "평행추세_참고"]
final_table = coef_table[cols_order].sort_values("p_raw", kind="stable")
print(final_table.to_string(index=False))


 Type  계층      coef       se    p_raw  p_adj_bh  유의_BH보정후  수신가구수  표본주의                 평행추세_참고
TypeC 고지출 -6.252591 1.756420 0.000371  0.003341      True    295 False  5단계 p=0.613 (미수신 49가구)
TypeB 고지출 -2.429732 0.965271 0.011832  0.053243     False    628 False  5단계 p=0.613 (미수신 49가구)
TypeC 저지출  9.675478 6.939703 0.163253  0.489758     False     28  True 5단계 p=0.764 (미수신 624가구)
TypeA 중지출  0.819589 0.687613 0.233289  0.524901     False    554 False 5단계 p=0.053 (미수신 243가구)
TypeA 저지출  1.056302 1.431923 0.460710  0.805521     False    187 False 5단계 p=0.764 (미수신 624가구)
TypeB 저지출  1.442018 2.335870 0.537014  0.805521     False     90 False 5단계 p=0.764 (미수신 624가구)
TypeB 중지출  0.417724 1.362253 0.759116  0.833748     False    305 False 5단계 p=0.053 (미수신 243가구)
TypeC 중지출 -1.219311 4.360564 0.779768  0.833748     False     74 False 5단계 p=0.053 (미수신 243가구)
TypeA 고지출 -0.162506 0.774218 0.833748  0.833748     False    772 False  5단계 p=0.613 (미수신 49가구)


In [5]:
print("="*60)
print("보정 전후 비교")
print("="*60)
n_sig_raw = (coef_table["p_raw"] < 0.05).sum()
n_sig_bh = coef_table["유의_BH보정후"].sum()
print(f"raw p<0.05 로 유의했던 조합: {n_sig_raw}개")
print(f"BH 보정 후에도 유의한 조합: {n_sig_bh}개")
print()
print("[BH 보정 후에도 살아남은 조합]")
survivors = final_table[final_table["유의_BH보정후"]]
print(survivors.to_string(index=False) if len(survivors) > 0 else "없음")
print()
print("[보정 전엔 유의했으나 보정 후 탈락한 조합]")
dropped = final_table[(final_table["p_raw"] < 0.05) & (~final_table["유의_BH보정후"])]
print(dropped.to_string(index=False) if len(dropped) > 0 else "없음")


보정 전후 비교
raw p<0.05 로 유의했던 조합: 2개
BH 보정 후에도 유의한 조합: 1개

[BH 보정 후에도 살아남은 조합]
 Type  계층      coef      se    p_raw  p_adj_bh  유의_BH보정후  수신가구수  표본주의                평행추세_참고
TypeC 고지출 -6.252591 1.75642 0.000371  0.003341      True    295 False 5단계 p=0.613 (미수신 49가구)

[보정 전엔 유의했으나 보정 후 탈락한 조합]
 Type  계층      coef       se    p_raw  p_adj_bh  유의_BH보정후  수신가구수  표본주의                평행추세_참고
TypeB 고지출 -2.429732 0.965271 0.011832  0.053243     False    628 False 5단계 p=0.613 (미수신 49가구)


In [6]:
# --- source CSV: the full nine-row table with Benjamini-Hochberg adjustment
_ci = result.conf_int()
_bh = coef_table.copy()
_bh["ci_low"] = [_ci.loc[f"{r.Type}_x_{r.계층}", "lower"] for r in _bh.itertuples()]
_bh["ci_high"] = [_ci.loc[f"{r.Type}_x_{r.계층}", "upper"] for r in _bh.itertuples()]
_bh["bh_rank"] = _bh["p_raw"].rank(method="first").astype(int)
_bh["bh_threshold"] = 0.05 * _bh["bh_rank"] / len(_bh)
_bh["pretrend_p"] = _bh["계층"].map(lambda t: _pt.loc[TIER_EN[t], "p_value"])
_bh["pretrend_n_control"] = _bh["계층"].map(lambda t: int(_pt.loc[TIER_EN[t], "n_never_receive"]))
_bh["계층"] = _bh["계층"].map(TIER_EN)
_bh.rename(columns={"Type": "type", "계층": "tier", "t": "t_stat", "유의_BH보정후": "reject_bh", "수신가구수": "n_recipients",
                    "표본주의": "small_cell"}) \
    [["type", "tier", "coef", "se", "ci_low", "ci_high", "t_stat", "p_raw", "p_adj_bh", "reject_bh", "bh_rank", "bh_threshold",
      "n_recipients", "small_cell", "pretrend_p", "pretrend_n_control"]] \
    .sort_values("p_raw", kind="stable").to_csv(OUT_DIR / "fe_bh_table.csv", index=False)

## 결과 해석 가이드

**확인할 순서**

1. `유의_BH보정후 = True`인 조합만 "통계적으로 안정적인 신호"로 인정
2. 그중 `표본주의 = True`(저지출×TypeC, n=28)면 → 표본이 작아 계수가 불안정할 수 있으므로
   BH를 통과했더라도 신뢰도를 한 단계 낮춰서 서술
3. `평행추세_참고`가 "경계선"이나 "검정력 부족"인 계층에서 살아남은 조합은 →
   "캠페인 효과"와 "원래 있던 사전추세"를 완전히 분리하지 못했다는 단서를 반드시 병기
4. **고지출 계층에서 유의한 결과가 나왔다면 특히 주의** — 5단계에서 고지출 계층은
   미수신 가구가 49명뿐이라 평행추세 검정 자체의 힘이 약했던 계층임

## 다음 단계
여기서 최종 확정된 Type×계층 조합을 갖고 8단계(위약검정, 캠페인 미수신 916가구)로 진행.
위약검정에서 같은 패턴이 안 나타나야 비로소 "캠페인 효과일 가능성이 있다"고 말할 수 있음.